# MVPruner L2 variation finetune debug on Kaggle
Smoke workflow: real RepViT/T5, FP32, two GPUs, finetune only, 20 optimizer steps. Enable Internet and a 2-GPU accelerator before running.

In [ ]:
import os, json
GITHUB_REPO_URL = os.environ.get('GITHUB_REPO_URL', 'https://github.com/huyvanzzz/Group_HND_car.git')
GITHUB_BRANCH = os.environ.get('GITHUB_BRANCH', 'huy-v2drop')
os.environ['GITHUB_REPO_URL'] = GITHUB_REPO_URL
os.environ['GITHUB_BRANCH'] = GITHUB_BRANCH
os.environ['CONFIG'] = 'configs/repvit_t5_efficient_mini_kaggle_2gpu_mvpruner_l2.yaml'
os.environ['RUNTIME_CONFIG'] = '/tmp/mvpruner_finetune_debug.yaml'
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
except Exception as exc:
    print('HF_TOKEN was not loaded:', exc)
print(json.dumps({'repo': GITHUB_REPO_URL, 'branch': GITHUB_BRANCH, 'hf_token_present': bool(os.environ.get('HF_TOKEN'))}, indent=2))

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working
if [ ! -d Efficient_VLM_For_Autonomous_Driving/.git ]; then
  git clone --branch "$GITHUB_BRANCH" "$GITHUB_REPO_URL" Efficient_VLM_For_Autonomous_Driving
fi
cd Efficient_VLM_For_Autonomous_Driving
git fetch origin "$GITHUB_BRANCH"
git checkout "$GITHUB_BRANCH"
git pull --ff-only origin "$GITHUB_BRANCH"
python -m pip install -q -e ".[mvpruner]"
python -m pip install -q pycocoevalcap
python - <<'PY'
import efficient_vlm_ad, torch, tokenizers, transformers
assert transformers.__version__ == "4.46.3", transformers.__version__
assert tokenizers.__version__ == "0.20.3", tokenizers.__version__
assert torch.cuda.is_available(), 'Kaggle GPU is not enabled'
assert torch.cuda.device_count() >= 2, 'Select a Kaggle 2-GPU accelerator'
print('package', efficient_vlm_ad.__file__)
print('torch', torch.__version__, 'transformers', transformers.__version__, 'tokenizers', tokenizers.__version__)
print('gpus', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
PY
python -m efficient_vlm_ad --help >/dev/null
nvidia-smi

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
python - <<'PY'
import os
from pathlib import Path
import yaml
cfg = yaml.safe_load(Path(os.environ['CONFIG']).read_text())
cfg['project']['output_dir'] = 'outputs/mvpruner_finetune_debug'
cfg['cache']['dir'] = 'outputs/mvpruner_finetune_debug/cache'
assert cfg['runtime']['precision'] == 'fp32'
Path(os.environ['RUNTIME_CONFIG']).write_text(yaml.safe_dump(cfg, sort_keys=False))
print(yaml.safe_dump({'output': cfg['project']['output_dir'], 'cache': cfg['cache']['dir'], 'precision': cfg['runtime']['precision'], 'fusion': cfg['model']['fusion']}))
PY
python -m efficient_vlm_ad inspect-data --config "$RUNTIME_CONFIG" --debug --debug-samples 2
python -m efficient_vlm_ad prepare-data --config "$RUNTIME_CONFIG" --subset smoke --debug --debug-samples 2
python -m efficient_vlm_ad prepare-features --config "$RUNTIME_CONFIG" --subset smoke --debug --debug-samples 2

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
python -m efficient_vlm_ad diagnose-train --config "$RUNTIME_CONFIG" --stage finetune --debug --debug-samples 1 --debug-numerics
PYTHONUNBUFFERED=1 accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision no --dynamo_backend no -m efficient_vlm_ad train --config "$RUNTIME_CONFIG" --stage finetune --max-steps 20 --debug --debug-samples 1 --debug-numerics --progress-log-every-steps 1
tail -n 30 outputs/mvpruner_finetune_debug/debug/train_progress.jsonl || true
ls -lh outputs/mvpruner_finetune_debug/checkpoints

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
PROFILE=outputs/mvpruner_finetune_debug
CHECKPOINT="$PROFILE/checkpoints/finetune_best.pt"
if [ ! -f "$CHECKPOINT" ]; then CHECKPOINT="$PROFILE/checkpoints/finetune_latest.pt"; fi
python -m efficient_vlm_ad evaluate --config "$RUNTIME_CONFIG" --checkpoint "$CHECKPOINT" --max-samples 32 --batch-size 16 --debug --debug-samples 1
python -m efficient_vlm_ad benchmark --config "$RUNTIME_CONFIG" --checkpoint "$CHECKPOINT" --max-samples 32 --debug --debug-samples 1
cat "$PROFILE/metrics.json" || true
cat "$PROFILE/benchmark.json" || true
head -n 10 "$PROFILE/predictions.jsonl" || true